# State of Data Brasil 2025 — Bronze
### Tech Challenge Fase 3 — Grupo 6
### Edição 2025

Fonte: [Kaggle — State of Data Brasil](https://www.kaggle.com/datahackers/datasets)

Esta é a camada **Bronze** da edição 2025 — a porta de entrada do dado no Data
Lake. Aqui **não se toma nenhuma decisão de negócio**: só se ingere o dado como
ele veio, resolve o mínimo pra ele ser utilizável (o nome das colunas), registra
metadado de ingestão e exporta em Parquet.

**Formato do cabeçalho desta base:** os nomes de coluna vêm como
`codigo_descricao` numa string só, com código numérico pontuado (`1.a`,
`8.d.11`). Pra o dado ser referenciável no Spark eu preciso separar código e
descrição — é o único "conserto" que a Bronze faz, e ele é técnico, não de
negócio. Esse parser (`extrair_codigo_e_descricao_2025`) fica em
`utils/functions.py`, por ser o único helper de nome de coluna que depende do
formato do CSV; o resto dos utils é compartilhado.

Todas as 388 colunas originais continuam aqui, sem seleção — quem decide o que
interessa é a Silver.

## 1. Preparando o ambiente e carregando a base

In [1]:
from pyspark.sql import SparkSession, functions as F
import re
import sys
from pathlib import Path
from collections import Counter

# Bootstrap: sobe a partir do diretório atual até achar utils/config.py —
# funciona não importa de onde o notebook seja executado (local, cluster, CI)
CAMINHO_ATUAL = Path.cwd().resolve()

for caminho in [CAMINHO_ATUAL, *CAMINHO_ATUAL.parents]:
    if (caminho / "utils" / "config.py").exists():
        RAIZ_PROJETO = caminho
        break
else:
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto (utils/config.py)."
    )

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from utils.config import CAMINHO_RAW, CAMINHO_BRONZE, CAMINHO_BRONZE_METADADOS
# Parser do cabeçalho desta base — mora em utils/functions.py por ser o único
# helper de nome de coluna que depende do formato do CSV. O resto
# (col_segura, slug, obter_bloco...) é compartilhado entre as edições.
from utils.functions import extrair_codigo_e_descricao_2025

spark = SparkSession.builder.appName("state-of-data-2025-bronze").getOrCreate()

# Bronze é um caminho ÚNICO particionado por ano_pesquisa, escrito pelos 3
# integrantes. O overwrite padrão do Spark ("static") apagaria a pasta inteira
# (inclusive as partições dos outros anos). Com "dynamic", o overwrite só afeta
# a partição que este notebook realmente escreve (ano_pesquisa=2025).
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

c:\Users\Henrique\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


A pesquisa tem perguntas abertas (texto livre) que podem conter vírgula e quebra
de linha dentro da resposta. Sem `multiLine` + `quote`/`escape`, o Spark quebraria
as colunas no lugar errado.

In [2]:
# Em produção (AWS), esse caminho vira algo tipo:
# "s3://<bucket-do-grupo>/raw/state_of_data_2025_2026.csv"
CAMINHO_CSV_ORIGEM = CAMINHO_RAW / "state_of_data_2025_2026.csv"

df = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(str(CAMINHO_CSV_ORIGEM))
)

print(f"{df.count()} linhas, {len(df.columns)} colunas")

3495 linhas, 388 colunas


## 2. Resolvendo o nome das colunas

Os nomes vêm no formato `codigo_descricao` — ex: `1.a_idade`, `1.b_genero`,
`8.d.11_Criando e mantendo a infra...`. O código tem 1 ou 2 níveis: o "pai" de
uma pergunta (`8.d`) e as opções de um bloco de múltipla escolha (`8.d.1`,
`8.d.2`, ...).

Uma pegadinha do CSV de 2025: o separador entre código e descrição é
**inconsistente** — a maioria usa `_` (`1.a_idade`), mas ~30 colunas de opção de
bloco usam **espaço** (`3.f.1 Colaboradores usando AI...`). O parser precisa
aceitar os dois.

In [3]:
df.columns[:6]

['0.a_token',
 '0.d_data/hora_envio',
 '1.a_idade',
 '1.a.1_faixa_idade',
 '1.b_genero',
 '1.c_cor/raca/etnia']

O parser (`extrair_codigo_e_descricao_2025`, importado de `utils/functions.py`)
captura o código (`\d+.letra` com um nível opcional `.n`) seguido de `_` **ou**
espaço, e o resto como descrição. Testo contra as 388 colunas antes de aplicar —
o esperado é 0 falha.

In [4]:
pares_teste = [extrair_codigo_e_descricao_2025(c) for c in df.columns]
sem_codigo = [nome for codigo, nome in pares_teste if codigo is None]
print(f"Colunas sem código identificado: {len(sem_codigo)} (esperado: 0)")

Colunas sem código identificado: 0 (esperado: 0)


Cada opção de um bloco de múltipla escolha já tem uma descrição própria, então a
maioria dos nomes de coluna é única. Ainda assim, **43 descrições se repetem**
entre colunas diferentes (ex: "Databricks" aparece nas ferramentas de ETL de mais
de um papel; "Remuneração/Salário" aparece em critério de escolha e em motivo de
insatisfação). Quando a descrição se repete, adiciono o código como sufixo
(`descricao__codigo`) pra garantir nome único e sem ambiguidade — do contrário o
Spark não conseguiria distinguir as duas colunas.

In [5]:
pares = [extrair_codigo_e_descricao_2025(c) for c in df.columns]
contagem = Counter(descricao for _, descricao in pares)

# Só coloco o código junto do nome quando a descrição se repete em mais de uma
# coluna (blocos de múltipla escolha e opções com rótulo idêntico entre blocos).
novos_nomes, mapa_codigo_para_nome = [], {}
for codigo, descricao in pares:
    nome_final = f"{descricao}__{codigo}" if (contagem[descricao] > 1 and codigo) else descricao
    novos_nomes.append(nome_final)
    if codigo:
        mapa_codigo_para_nome[codigo] = nome_final

assert len(novos_nomes) == len(set(novos_nomes)), "Ainda há nomes duplicados!"
df = df.toDF(*novos_nomes)
print("Renomeação concluída, sem duplicatas.")
print("\nExemplo de nomes finais:")
for c in df.columns[:6]:
    print(" -", c)

Renomeação concluída, sem duplicatas.

Exemplo de nomes finais:
 - token
 - data/hora_envio
 - idade
 - faixa_idade
 - genero
 - cor/raca/etnia


**Importante pra Silver:** pra montar os blocos de múltipla escolha depois
(ex: "todas as ferramentas de ETL que a pessoa marcou"), a Silver precisa saber
qual código (`4.d.1`, `4.d.2`...) virou qual nome de coluna. Vou salvar o
`mapa_codigo_para_nome` como uma tabelinha ao lado do Parquet — assim a Silver
só carrega o de-para pronto.

Escrevo com Python puro (não via Spark): a tabela é pequena (388 linhas) e criar
DataFrame Spark a partir de lista Python pura costuma dar problema de worker no
Spark local no Windows. No cluster (AWS Glue/EMR), isso vira um upload `boto3`
pro S3.

In [6]:
import csv as csv_module
import os

CAMINHO_MAPA_COLUNAS = CAMINHO_BRONZE_METADADOS / "mapa_colunas_2025.csv"
os.makedirs(CAMINHO_MAPA_COLUNAS.parent, exist_ok=True)

with open(CAMINHO_MAPA_COLUNAS, "w", newline="", encoding="utf-8") as f:
    escritor = csv_module.writer(f)
    escritor.writerow(["codigo", "nome_coluna"])
    for codigo, nome in mapa_codigo_para_nome.items():
        escritor.writerow([codigo, nome])

print(f"Mapa de colunas salvo em: {CAMINHO_MAPA_COLUNAS} ({len(mapa_codigo_para_nome)} códigos)")

Mapa de colunas salvo em: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\bronze\metadados\mapa_colunas_2025.csv (388 códigos)


## 3. Auditoria rápida (só registrar, sem corrigir nada aqui)

A Bronze não corrige qualidade — mas vale registrar o que se encontra, pra
conferir depois que a Silver tratou tudo. Duas checagens: duplicidade de registro
e volume de nulo por coluna (só contando).

In [7]:
col_id = df.columns[0]
total = df.count()
distintos = df.select(col_id).distinct().count()

print(f"Total de linhas: {total}")
print(f"IDs distintos: {distintos}")
print("Sem duplicidade" if total == distintos else "ATENÇÃO: existem IDs duplicados (tratamento fica pra Silver)")

Total de linhas: 3495
IDs distintos: 3494
ATENÇÃO: existem IDs duplicados (tratamento fica pra Silver)


In [8]:
# Contagem de nulo por coluna, só como registro de auditoria
total_linhas = df.count()
nulos_por_coluna = df.select([
    F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c) for c in df.columns
]).collect()[0].asDict()

colunas_com_mais_nulo = sorted(nulos_por_coluna.items(), key=lambda x: -x[1])[:10]
print("Top 10 colunas com mais nulo (informativo, não tratado aqui):")
for nome, qtd in colunas_com_mais_nulo:
    print(f"  {qtd:5d} ({qtd/total_linhas*100:5.1f}%)  {nome}")

Top 10 colunas com mais nulo (informativo, não tratado aqui):
   3373 ( 96.5%)  pais_onde_mora
   3312 ( 94.8%)  oportunidade_buscada
   3308 ( 94.6%)  experiencia_em_processos_seletivos
   3303 ( 94.5%)  tempo_em_busca_de_oportunidade
   3081 ( 88.2%)  objetivo_na_area_de_dados
   3080 ( 88.1%)  tecnologia_data_lake
   3075 ( 88.0%)  tecnologia_data_warehouse
   3005 ( 86.0%)  Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.__8.a.1
   3005 ( 86.0%)  Sou responsável pela coleta e limpeza dos dados que uso para análise e modelagem.
   3005 ( 86.0%)  Sou responsável por entrar em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.


## 4. Adicionando metadados de ingestão

Colunas de controle, padrão de qualquer Bronze de Data Lake: de onde veio cada
linha, quando chegou, e o ano da pesquisa (que também é a chave de partição).

In [9]:
df_bronze = (
    df
    .withColumn("dt_ingestao", F.current_timestamp())
    .withColumn("arquivo_origem", F.input_file_name())
    .withColumn("fonte_pesquisa", F.lit("State of Data Brasil - Data Hackers + Bain"))
    .withColumn("ano_pesquisa", F.lit(2025))
)

print(f"Colunas finais (com metadados): {len(df_bronze.columns)}")
df_bronze.select("dt_ingestao", "arquivo_origem", "fonte_pesquisa", "ano_pesquisa").show(3, truncate=False)

Colunas finais (com metadados): 392
+--------------------------+---------------------------------------------------------------------------------------------------+------------------------------------------+------------+
|dt_ingestao               |arquivo_origem                                                                                     |fonte_pesquisa                            |ano_pesquisa|
+--------------------------+---------------------------------------------------------------------------------------------------+------------------------------------------+------------+
|2026-08-23 05:03:54.191692|file:///C:/Users/Henrique/Desktop/Tech%20Challenge%203%20local/data/raw/state_of_data_2025_2026.csv|State of Data Brasil - Data Hackers + Bain|2025        |
|2026-08-23 05:03:54.191692|file:///C:/Users/Henrique/Desktop/Tech%20Challenge%203%20local/data/raw/state_of_data_2025_2026.csv|State of Data Brasil - Data Hackers + Bain|2025        |
|2026-08-23 05:03:54.191692|file:///C:/

## 5. Exportando para Parquet

Particionado por `ano_pesquisa`. Com `partitionOverwriteMode = dynamic` (setado
lá em cima), escrever a partição 2025 afeta só essa partição — o caminho é
compartilhado por várias edições e não pode ser apagado por inteiro a cada
escrita.

Em produção, troca a raiz local pelo bucket S3 do grupo
(`s3://<bucket>/bronze/state_of_data/`).

In [10]:
df_bronze.write \
    .mode("overwrite") \
    .partitionBy("ano_pesquisa") \
    .parquet(str(CAMINHO_BRONZE))

print(f"Bronze exportada com sucesso em: {CAMINHO_BRONZE}")
print(f"Linhas: {df_bronze.count()} | Colunas: {len(df_bronze.columns)}")

Bronze exportada com sucesso em: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\bronze\state_of_data
Linhas: 3495 | Colunas: 392


## 6. Conclusão

A Bronze 2025 ficou com as 388 colunas originais (renomeadas só pra virarem
utilizáveis, sem decisão de negócio) + 4 colunas de metadado de ingestão,
particionada por `ano_pesquisa=2025`. O mapa código → nome de coluna
(`mapa_colunas_2025.csv`) ficou salvo ao lado, pra Silver reconstruir os blocos
de múltipla escolha sem refazer esse trabalho.